In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Importações dos seus ficheiros .py
from linhas import LINHAS, LOCAIS, TODAS_ESTACOES, CORES
from IA import interpretar_pedido, narrar
from main import planejar

def desenhar_linhas_html(resultado):
    """Desenha as três linhas do metropolitano em HTML."""
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"] or [])
    bloqueadas = set(resultado["bloqueadas"] or [])
    
    html_total = ""
    for nome_linha, estacoes in LINHAS.items():
        cor_linha = CORES[nome_linha]
        linhas_html = [f"<h4 style='color:{cor_linha}; margin-top:15px;'>{nome_linha}</h4>"]
        
        for estacao in estacoes:
            if estacao in bloqueadas:
                cor, marca = "#d32f2f", "bloqueada"
            elif estacao in (resultado.get("origem"), resultado.get("destino")) and estacao in caminho:
                cor, marca = "#0d47a1", ("origem" if estacao == resultado["origem"] else "destino")
            elif estacao in caminho:
                cor, marca = cor_linha, "rota"
            elif estacao in visitados:
                cor, marca = "#9e9e9e", "visitada pela busca"
            else:
                cor, marca = "#e0e0e0", ""
                
            linhas_html.append(
                f"<div style='display: flex;align-items:center; gap:8px; font-family: sans-serif; font-size:13px'>"
                f"<span style='display:inline-block;width: 14px;height:14px;border-radius: 50%; background: {cor}'></span>"
                f"<span style='min-width:150px'> {estacao}</span><span style='color:#666'> {marca}</span></div>"
            )
        html_total += f"<div style='border-left: 4px solid {cor_linha}; padding-left: 8px; margin-bottom: 10px;'>" + "".join(linhas_html[1:]) + "</div>"
    
    return html_total

# Preparar opções para os Dropdowns
opcoes = [(f"📍 {local}", ("local", local)) for local in LOCAIS] + \
         [(f"🚉 {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES]

# Widgets da Interface[cite: 1]
txt_pedido = widgets.Textarea(placeholder="Ex.: Estou no MASP e quero ir para o Tucuruvi...", layout=widgets.Layout(width="95%", height="60px"))
btn_interpretar = widgets.Button(description="🧠 Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, description="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")

sel_fechadas = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Fechadas:")
sel_manut = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Elevador ❌:")
sel_paralisadas = widgets.SelectMultiple(options=list(LINHAS.keys()), description="Paralisadas:")
rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")

btn_buscar = widgets.Button(description="🚇 Buscar rota", button_style="success")
saida = widgets.Output()

def ao_interpretar(_):
    with saida:
        clear_output()
        pedido, msg = interpretar_pedido(txt_pedido.value)
        print(msg)
        if pedido:
            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]
            print("✅ Campos preenchidos! Confira e clique em 'Buscar rota'.")

def ao_buscar(_):
    with saida:
        clear_output()
        pedido = {"origem": dd_origem.value, "destino": dd_destino.value, "acessibilidade": chk_acess.value}
        r = planejar(pedido, sel_fechadas.value, sel_manut.value, sel_paralisadas.value, rb_algoritmo.value)
        
        display(HTML(f"<h3>{r['algoritmo']}: {r['origem']} ➔ {r['destino']}</h3>"))
        print("🗣️ Narrador:", narrar(r))
        print(f"\n📊 DADOS TÉCNICOS:")
        print(f"Paradas: {r['paradas']} | Tempo: ~{r['tempo_min']} min | Baldeações: {r['qtd_baldeacoes']}")
        print(f"Estações visitadas pelo algoritmo: {len(r['visitados'])}")
        print(f"Regras disparadas: {', '.join(r['regras_usadas'])}")
        
        display(HTML(desenhar_linhas_html(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    widgets.HTML("<h2>🚇 MetrôBot SP 2.0 (Linhas 1, 2 e 3)</h2>"),
    txt_pedido, btn_interpretar,
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HBox([sel_fechadas, sel_manut, sel_paralisadas]),
    btn_buscar, saida
])

display(painel)